# Purdue University Affiliation Analysis

This notebook analyzes awardee names for Purdue University connections using the Institution Checker pipeline.

**Workflow:**
1. **Setup**: Configure environment, reload modules, and set API keys.
2. **Data Loading**: Load and prepare the dataset (Full or Sample).
3. **Clustering**: Group similar names to optimize processing.
4. **Validation**: Run a quick test on known VIPs to ensure accuracy.
5. **Execution**: Run the full pipeline on the clustered dataset.
6. **Analysis**: Review results, verify specific targets, and export data.

In [1]:
# 1. SETUP & CONFIGURATION
# -----------------------------------------------------------------------------
# This cell handles module reloading and API configuration.
# Run this FIRST to ensure the environment is ready.

import sys
import importlib
import pandas as pd
import time
from pathlib import Path

# --- Fresh-run state reset (avoid stale notebook variables) ---
CURRENT_RUN_ID = None
RUN_RESULTS_READY = False
for var_name in [
    "df", "df_base", "df_vips", "df_others", "df_random", "df_clustered",
    "cluster_reps", "cluster_results", "cluster_institution_map", "final_results"
]:
    if var_name in globals():
        del globals()[var_name]

# --- Force Local Source Path (prevents stale UNC/site-package imports) ---
workspace_src = Path(r"e:\Awards DB Code\step3_institution_checker\src")
if workspace_src.exists():
    sys.path = [p for p in sys.path if "institution_checker" not in p.lower()]
    sys.path.insert(0, str(workspace_src))
    print(f"Using local source path: {workspace_src}")

# Drop cached institution_checker modules so imports resolve from the path above.
for mod in [m for m in list(sys.modules) if m == "institution_checker" or m.startswith("institution_checker.")]:
    del sys.modules[mod]

# --- Module Reloading (Development Mode) ---
# Reinstall local package to ensure latest changes are applied
!pip uninstall -y institution_checker
!pip install -e .

# Force reload of modules to pick up changes
modules_to_reload = [
    'institution_checker.config',
    'institution_checker.search',
    'institution_checker.llm_processor',
    'institution_checker.main',
    'institution_checker'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"Reloaded {module_name}")

# --- Imports (AFTER Reloading) ---
# Import AFTER reloading to ensure we get the new function objects
import institution_checker
import institution_checker.llm_processor
import institution_checker.search
from institution_checker import run_pipeline, resolve_names, set_api_key
from institution_checker.llm_processor import enable_prompt_logging
from name_matcher.pipeline import NameMatcher, NameMatcherConfig

print(f"institution_checker loaded from: {institution_checker.__file__}")

# --- API Configuration ---
API_KEY = "sk-3f1dbbf2450e46ab9541dffba4f18ec6"
set_api_key(API_KEY)

# Configure NameMatcher
config = NameMatcherConfig(name_column="name", use_tqdm=True)
config.llm_token = API_KEY

print("\nSetup complete:")
print("   - Fresh in-memory state initialized")
print("   - Modules reloaded and installed")
print("   - API keys configured")
print("   - Browser timeout fixes active (15s)")
print("   - Search strategies optimized (DuckDuckGo prioritized + Rate Limit Protection)")
print("   - Evidence gathering expanded (8 excerpts)")

Using local source path: e:\Awards DB Code\step3_institution_checker\src
Found existing installation: institution-checker 0.1.0
Uninstalling institution-checker-0.1.0:
  Successfully uninstalled institution-checker-0.1.0
Obtaining file:///E:/Awards%20DB%20Code/step3_institution_checker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for institution-checker (pyproject.toml): started
  Building editable for institution-checker (pyproject.toml): finished with status 'done'
  Created wheel for institution-checker: filename=institu

In [2]:
# 2. DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
# Select the dataset to process and prepare it for analysis.

# --- Configuration ---
DATASET_OPTION = "nobel"  # Options: "purdue", "nobel", "sample", "accuracy_test_25"

# File Paths
PATHS = {
    "purdue": r"E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx",
    "nobel": r"E:\Ace\Nobel Only Test Set.xlsx"
}

# --- Load Data ---
if DATASET_OPTION == "sample":
    print("ðŸ“‰ CREATING TEST SAMPLE: 9 VIPs + 100 Random Names")
    # Load base dataset (Nobel) for sampling
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].copy()
    df_others = df_base[~vip_mask].copy()
    
    sample_size = min(100, len(df_others))
    df_random = df_others.sample(n=sample_size, random_state=42)
    
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    print(f"âœ… Sample created: {len(df)} records")

elif DATASET_OPTION == "accuracy_test_25":
    print("ðŸ“‰ CREATING ACCURACY TEST: 9 VIPs + 16 Random Non-Connected")
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    # 1. Get VIPs
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].drop_duplicates(subset=['name']).copy()
    
    # 2. Get Others (Non-VIPs)
    df_others = df_base[~vip_mask].copy()
    
    # 3. Select 16 random names
    # Using a fixed seed ensures reproducibility. 
    df_random = df_others.sample(n=16, random_state=123) 
    
    # Combine
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    
    print(f"âœ… Test set created: {len(df)} records")
    print(f"   VIPs (Positive Controls): {len(df_vips)}")
    print(f"   Others (Negative Controls): {len(df_random)}")
    
    print("\nðŸ“‹ List of names to check:")
    print("-" * 30)
    for i, name in enumerate(df['name'], 1):
        role = "VIP" if any(v in name for v in vip_names) else "Other"
        print(f"{i:2}. {name:<30} [{role}]")

elif DATASET_OPTION in PATHS:
    path = PATHS[DATASET_OPTION]
    df = pd.read_excel(path, dtype=str)
    print(f"âœ… Loaded {len(df):,} records from {DATASET_OPTION.upper()} dataset")
    print(f"   Source: {path}")

else:
    raise ValueError(f"Invalid DATASET_OPTION: {DATASET_OPTION}")

# --- Clustering ---
print("\nðŸ§© Clustering similar names...")
matcher = NameMatcher(config)
result = matcher.cluster(df)
df_clustered = result.dataframe
print(f"âœ… Clustering complete: {len(df)} names -> {df_clustered['cluster_label'].nunique()} clusters")


âœ… Loaded 1,014 records from NOBEL dataset
   Source: E:\Ace\Nobel Only Test Set.xlsx

ðŸ§© Clustering similar names...
--- Name Matcher Process Started (v6.8.8 Improved Canonical Name Selection) ---

1. Loading and preparing data...
   Loaded 1014 names. Done in 0.03s
2. Vectorizing names for similarity scoring...
   Done in 0.02s
3. Generating candidate pairs via Blocking...
   Generated 18 candidate pairs from blocking.
   Done in 0.00s
4. Applying Stricter Cluster-Centric Filtering...


   Filtering Pairs: 100%|██████████| 18/18 [00:00<?, ?pair/s]

   Filter Stats: {'merged_expansion': 8, 'rejected_transitive_conflict': 9}
   Done in 0.00s
5. Reviewing ambiguous pairs with LLM...
   Done in 0.00s
6. Finalizing cluster data structures...
   Done in 0.00s
7. Refining clusters by ejecting outliers...
   Found and ejected 0 outliers from clusters.
   Done in 0.01s
8. Assigning canonical names and saving results...

--- Results Summary ---
   - Total names processed: 1014
   - Unique entities (clusters) found: 1006

   --- Sample of Largest Clusters Found ---
   Cluster 1 (Size: 3): 'International Committee Of The Red Cross'
     - International Committee of the Red Cross
     - International Committee of the Red Cross
     - International Committee of the Red Cross
   Cluster 2 (Size: 2): 'Frederick Sanger'
     - Frederick Sanger
     - Frederick Sanger
   Cluster 3 (Size: 2): 'K Barry Sharpless'
     - K. Barry Sharpless
     - K. Barry Sharpless
   Cluster 4 (Size: 2): 'Linus Pauling'
     - Linus Pauling
     - Linus Pauling
   C

In [3]:
# 3. PIPELINE VALIDATION (VIP TEST)
# -----------------------------------------------------------------------------
# Run a quick test on known VIPs to verify pipeline accuracy before the full run.

vip_test_names = [
    "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
    "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
    "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
]

print("ðŸ§ª VIP ACCURACY TEST (9 names)")
print("=" * 60)
print("Testing pipeline accuracy...")

start_time = time.time()

# Run pipeline on VIPs
vip_results = await run_pipeline(
    vip_test_names, 
    batch_size=3,  # Reduced to avoid DDG rate limits (was 9)
    use_enhanced_search=True,
    dataset_profile="high_connection",
    use_dynamic_batching=False
)

elapsed = time.time() - start_time

# Analyze Results
print(f"\nðŸ“Š VIP TEST RESULTS ({elapsed:.1f}s, {elapsed/9:.1f}s per name):")
print("-" * 60)

vip_connected = 0
for name, res in zip(vip_test_names, vip_results):
    verdict = res.get('verdict', 'unknown')
    institution = res.get('institution', '')
    is_purdue = 'purdue' in institution.lower() if institution else False
    is_connected = verdict == 'connected' and is_purdue
    
    if is_connected:
        vip_connected += 1
        icon = "âœ…"
    else:
        icon = "âŒ"
    
    print(f"   {icon} {name}: {verdict}")

recall = (vip_connected / 9) * 100
print(f"\nðŸ† VIP RECALL: {vip_connected}/9 ({recall:.0f}%)")

if vip_connected >= 8:
    print("âœ… Accuracy OK - proceed with full dataset")
else:
    print("âš ï¸  Low accuracy - check module reload (Cell 1) before proceeding")

ðŸ§ª VIP ACCURACY TEST (9 names)
Testing pipeline accuracy...
[PIPELINE] Starting: 9 name(s) in 3 batch(es) using enhanced search
[PIPELINE] Batch size: 3, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/3 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/3 =====
[PIPELINE] Names in this batch: ['Akira Suzuki', 'Ei-ichi Negishi', 'Herbert C. Brown']
[BATCH] Processing 3 names: Akira Suzuki, Ei-ichi Negishi, Herbert C. Brown
[BATCH] Phase 1: Running searches in parallel for all 3 names (max 8 concurrent)
[BATCH] Phase 1 completed in 131.3s
[BATCH] Search succeeded for Akira Suzuki: 20 results (mode=basic_only, queries=1, bucket=plausible)
[BATCH] Search succeeded for Ei-ichi Negishi: 19 results (mode=basic_only, queries=1, bucket=plausible)
[BATCH] Search succeeded for Herbert C. Brown: 27 results (mode=basic_plus_rescue_plus_enhanced, queries=3, bucket=plausible)
[BATCH] Search telemetry: avg_queries=1.67, avg_attempts=1.67, rescue=1, enhanced=1, slow_fallback=0
[BATCH] Phase 2: Evaluating search results and running LLM analysis in parallel for all 3 names (max 6 concurrent)
[BATCH] Survey counts: hard_no=0, borderline=0, plausible=3, rescue_used=1, rescue_promoted=0
[PROGRESS] Starting LLM analysis for: Akira 

In [4]:
# 4. MAIN EXECUTION (FULL DATASET)
# -----------------------------------------------------------------------------
# Run the affiliation checker on the full clustered dataset using the optimized library pipeline.

TEST_MODE = False
if DATASET_OPTION == "accuracy_test_25":
    DATASET_PROFILE = "high_connection"
    USE_DYNAMIC_BATCHING = False
    USE_ENHANCED_SEARCH = True
    BATCH_SIZE = 5
else:
    DATASET_PROFILE = "low_connection"
    USE_DYNAMIC_BATCHING = True
    USE_ENHANCED_SEARCH = False  # Throughput mode for large sparse Nobel-style runs
    BATCH_SIZE = 30

CURRENT_RUN_ID = int(time.time())
RUN_RESULTS_READY = False

cluster_reps = df_clustered.groupby('cluster_label')['name'].first().reset_index()
cluster_reps.columns = ['cluster_label', 'representative_name']

if TEST_MODE:
    cluster_reps = cluster_reps.head(20)
    print(f"TEST MODE: Checking {len(cluster_reps)} clusters")
else:
    print(f"Checking all {len(cluster_reps)} clusters")

names_to_check = cluster_reps['representative_name'].tolist()

print("Starting Optimized Institution Algorithm (internal library v2)...")
print(f"   Profile: {DATASET_PROFILE}")
print(f"   Search Batch: {BATCH_SIZE}")
print(f"   Dynamic batching: {USE_DYNAMIC_BATCHING}")
print(f"   Enhanced search: {USE_ENHANCED_SEARCH}")
print(f"   Names to process: {len(names_to_check)}")
print(f"   Run ID: {CURRENT_RUN_ID}")
print("-" * 60)

start_time = time.time()
cluster_results = await run_pipeline(
    names_to_check,
    batch_size=BATCH_SIZE,
    use_enhanced_search=USE_ENHANCED_SEARCH,
    dataset_profile=DATASET_PROFILE,
    use_dynamic_batching=USE_DYNAMIC_BATCHING,
)

elapsed = time.time() - start_time
avg_per_name = elapsed / len(names_to_check) if names_to_check else 0
print(f"\nTIMING: total={elapsed:.1f}s | avg={avg_per_name:.2f}s/name")
print(f"THROUGHPUT: {60/avg_per_name:.1f} names/min" if avg_per_name > 0 else "THROUGHPUT: N/A")

cluster_institution_map = {}
for cluster_label, rep_name, res in zip(cluster_reps['cluster_label'], cluster_reps['representative_name'], cluster_results):
    if not isinstance(res, dict):
        res = {'raw_result': res}

    institution = res.get('institution', '')
    verdict = res.get('verdict', '')
    purdue_match = (
        verdict == 'connected'
        and (
            'purdue university' in str(institution).lower()
            or (str(institution).lower().startswith('purdue') and 'university' in str(institution).lower())
        )
    )

    res['purdue_connection'] = purdue_match
    cluster_institution_map[cluster_label] = res

fields_to_map = [
    'institution', 'verdict', 'relationship_type', 'relationship_timeframe',
    'verification_detail', 'summary', 'primary_source', 'verification_status',
    'temporal_context', 'confidence', 'purdue_connection',
    'pre_llm_bucket', 'pre_llm_stage', 'pre_llm_score', 'pre_llm_summary',
    'pre_llm_reason_codes', 'pre_llm_used_rescue_query'
]

for field in fields_to_map:
    df_clustered[field] = df_clustered['cluster_label'].map(
        lambda x: cluster_institution_map.get(x, {}).get(field, None)
    )

final_results = df_clustered.copy()
final_results['_run_id'] = CURRENT_RUN_ID
final_results['_run_generated_at'] = pd.Timestamp.utcnow().isoformat()
RUN_RESULTS_READY = True

if 'pre_llm_stage' in final_results.columns:
    print("Pre-LLM stage distribution:")
    print(final_results['pre_llm_stage'].fillna('(missing)').value_counts().head(10))

print("Affiliation check complete")
print(f"Fresh rows: {len(final_results)}")

Checking all 1006 clusters
Starting Optimized Institution Algorithm (internal library v2)...
   Profile: low_connection
   Search Batch: 10
   Dynamic batching: True
   Names to process: 1006
   Run ID: 1775573751
------------------------------------------------------------
[PIPELINE] Starting DYNAMIC BATCHING mode
[PIPELINE] Total: 1006 names, Search batch: 10, LLM Flush Size: 5
[PIPELINE] Using enhanced search, Dataset profile: low_connection


Processing search batches:   0%|          | 0/101 [00:00<?, ?batch/s]


[PIPELINE] ===== SEARCH BATCH 1/101 =====
[PIPELINE] Names: A. Michael Spence, Aage N. Bohr, Aaron Ciechanover, Aaron Klug, Abdulrazak Gurnah, Abdus Salam, Abhijit Banerjee, Abiy Ahmed Ali, Ada E. Yonath, Adam G. Riess
[SEARCH] Running parallel searches for 10 names (max 8 concurrent)...
[SEARCH] Search succeeded for A. Michael Spence: 7 results (mode=basic_only, queries=1, bucket=plausible)
[SEARCH] Search succeeded for Aage N. Bohr: 37 results (mode=basic_plus_rescue_plus_enhanced, queries=3, bucket=borderline)
[SEARCH] Search succeeded for Aaron Ciechanover: 16 results (mode=basic_only_demoted, queries=1, bucket=hard_no)
[SEARCH] Search succeeded for Aaron Klug: 12 results (mode=basic_only_demoted, queries=1, bucket=hard_no)
[SEARCH] Search succeeded for Abdulrazak Gurnah: 13 results (mode=basic_only_demoted, queries=1, bucket=hard_no)
[SEARCH] Search succeeded for Abdus Salam: 19 results (mode=basic_only_demoted, queries=1, bucket=hard_no)
[SEARCH] Search failed for Abhijit Banerj

CancelledError: 

In [ ]:
# 5. ANALYSIS & EXPORT (FRESH RUN ONLY)
# -----------------------------------------------------------------------------
# Analyze results from this kernel's latest run only.

EXPORT_FILENAME = 'institution_check_results.xlsx'
VIP_TARGETS = [
    "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
    "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
    "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
]
RUN_FAILED_VIP_DIAGNOSTICS = True
FAILED_VIP_TOP_K = 5

if 'final_results' not in globals():
    raise RuntimeError("No in-memory results found. Run Section 4 first.")
if not globals().get('RUN_RESULTS_READY', False):
    raise RuntimeError("Results are not marked fresh. Re-run Section 4 in this kernel.")
if 'CURRENT_RUN_ID' not in globals() or CURRENT_RUN_ID is None:
    raise RuntimeError("Missing CURRENT_RUN_ID. Re-run Section 1 then Section 4.")
if '_run_id' not in final_results.columns:
    raise RuntimeError("Result metadata missing (_run_id). Re-run Section 4.")

run_ids = set(final_results['_run_id'].dropna().astype(int).unique().tolist())
if run_ids != {int(CURRENT_RUN_ID)}:
    raise RuntimeError(
        f"Stale/mixed results detected: expected run_id={CURRENT_RUN_ID}, found={sorted(run_ids)}. Re-run Section 4."
    )

def _normalize_lookup_name(value):
    return ' '.join(str(value).strip().lower().split())

def _build_exact_name_map(df):
    exact = {}
    for _, row in df.iterrows():
        key = _normalize_lookup_name(row.get('name', ''))
        if key and key not in exact:
            exact[key] = row
    return exact

exact_name_map = _build_exact_name_map(final_results)
print(f"RESULTS ANALYSIS (run_id={CURRENT_RUN_ID})")
print("=" * 70)

total_records = len(final_results)
purdue_records = final_results[final_results['purdue_connection'] == True]
percentage = (len(purdue_records) / total_records * 100) if total_records > 0 else 0

print(f"   Total records: {total_records:,}")
print(f"   Purdue-affiliated: {len(purdue_records):,} ({percentage:.1f}%)")